# 01 · ETL e engenharia de atributos

**Etapas 1 e 2 do enunciado** · **Features F1 e F2** (critérios 1 e 2) · dono: Gabriel

Este notebook lê os lotes incorporados em `data/raw`, valida, limpa, cria atributos, separa dev e teste em ordem cronológica e grava o contexto do ciclo para os notebooks seguintes.

| | |
|---|---|
| **Entradas** | `data/raw/*.csv.gz` + `data/raw/manifest.json` · `config/config.yaml` |
| **Saídas** | `data/processed/dev.parquet`, `teste.parquet`, `ciclo_atual.json` · `evidencias/etl/diagnostico_<cycle_id>.json` |
| **Regra de ouro** | O que aprende com os dados (limites de outliers, escalas, categorias) fica no `Pipeline` e é ajustado só no dev. |

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, resolver, salvar_contexto
from src.ingestao import preparar_dados
from src.preprocessamento import (
    colunas_modelagem, construir_preprocessador, diagnosticar, dividir_temporal,
    limpar_e_engenhar, lista_features, salvar_particoes, validar_dados_brutos,
)
from src.schema import COLUNAS_BRUTAS

cfg = carregar_config()
alvo = cfg["avaliacao"]["alvo"]
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Entrada e versionamento dos dados
Cada lote tem `sha256` no manifesto. Registrar de novo o mesmo conteúdo não duplica nada, e alterar um lote já registrado derruba o pipeline. A versão dos dados (`data_version`) resume o conteúdo dos lotes incorporados.

In [ ]:
r = preparar_dados(cfg)
df_raw, manifesto = r["df"], r["manifesto"]
data_version, cycle_id = r["data_version"], r["cycle_id"]

evid_dir = resolver("evidencias/etl", RAIZ)
evid_dir.mkdir(parents=True, exist_ok=True)
anteriores = sorted(evid_dir.glob("diagnostico_*.json"))
lotes_atuais = [l["lote_id"] for l in manifesto["lotes"]]
lotes_anteriores = json.loads(anteriores[-1].read_text())["lotes"] if anteriores else []
lotes_novos = [l for l in lotes_atuais if l not in lotes_anteriores]

print(f"cycle_id      : {cycle_id}")
print(f"data_version  : {data_version}")
print(f"lotes         : {len(lotes_atuais)} ({lotes_atuais[0]} ... {lotes_atuais[-1]})")
if anteriores:
    print(f"novos desde o ciclo anterior: {lotes_novos if lotes_novos else 'nenhum (mesma versão dos dados)'}")
else:
    print("primeiro ciclo registrado: todos os lotes são novos")
print(f"linhas brutas : {len(df_raw):,}")

In [ ]:
pd.DataFrame(manifesto["lotes"])[["lote_id", "n_linhas", "chegada_min", "chegada_max"]].tail(5)

## 2. Validação e diagnóstico **antes** do tratamento
A validação interrompe o fluxo (exceção) se faltar coluna, se o alvo não for 0/1 ou se houver mês inválido.

In [ ]:
validar_dados_brutos(df_raw)
diag_antes = diagnosticar(df_raw)
print(json.dumps({k: v for k, v in diag_antes.items() if k != "extremos"}, indent=2, ensure_ascii=False))
pd.DataFrame(diag_antes["extremos"]).T

### Decisão sobre duplicatas (evidência executável)
O dataset não tem ID de reserva. Antes de decidir se linhas idênticas são erro, medimos quem elas são.

In [ ]:
repetida = df_raw.duplicated(subset=COLUNAS_BRUTAS, keep="first")
em_grupos = df_raw.loc[repetida, "market_segment"].isin(["Groups", "Offline TA/TO"]).mean()
maior_grupo = df_raw.groupby(COLUNAS_BRUTAS, dropna=False).size().max()
print(f"Linhas idênticas a outra            : {repetida.sum():,} ({repetida.mean():.1%} do bruto)")
print(f"Taxa de cancelamento nelas / demais : {df_raw.loc[repetida, alvo].mean():.1%} / {df_raw.loc[~repetida, alvo].mean():.1%}")
print(f"Parte em Groups ou Offline TA/TO    : {em_grupos:.1%} | maior grupo idêntico: {maior_grupo} linhas")
print(f"Taxa de cancelamento do bruto       : {df_raw[alvo].mean():.1%}")
print(f"Estratégia adotada (config)         : {cfg['etl']['duplicatas']['estrategia']}")

Linhas idênticas concentradas em reservas de grupo e agências, com cancelamento muito acima da média, sugerem **várias reservas de um mesmo pedido cancelando juntas**, não erro de digitação. Remover todas mudaria a taxa de cancelamento de forma artificial. Por isso a estratégia padrão mantém cópias dentro do mesmo lote e remove apenas o que **reaparece em outro lote** (reentrega de dados). A opção `todas` fica na config para análise de sensibilidade.

## 3. Limpeza e engenharia de atributos

In [ ]:
df, rel = limpar_e_engenhar(df_raw, cfg)
diag_depois = diagnosticar(df)

display(pd.DataFrame({
    "antes": {
        "linhas": diag_antes["n_linhas"],
        "colunas": diag_antes["n_colunas"],
        "colunas com ausentes": len(diag_antes["nulos_por_coluna"]),
        "taxa de cancelamento": diag_antes["taxa_cancelamento"],
    },
    "depois": {
        "linhas": diag_depois["n_linhas"],
        "colunas": diag_depois["n_colunas"],
        "colunas com ausentes": len(diag_depois["nulos_por_coluna"]),
        "taxa de cancelamento": diag_depois["taxa_cancelamento"],
    },
}))
print(json.dumps({k: v for k, v in rel.items() if k != "colunas_removidas"}, indent=2, ensure_ascii=False))
print("colunas descartadas:", rel["colunas_removidas"])

**Por que cada coluna descartada saiu** (detalhes em `docs/etapa1_problema_dados_avaliacao.md`):
`reservation_status` e `reservation_status_date` entregam o alvo · `assigned_room_type` e `booking_changes` só existem depois da reserva · a nacionalidade correta (`country`) costuma ser conhecida só no check-in · ano/mês/dia de chegada foram trocados por atributos numéricos que generalizam para datas futuras · `company` virou o indicador `tem_empresa`.

In [ ]:
por_lote = df.groupby("lote_id").agg(reservas=(alvo, "size"), taxa=(alvo, "mean"))
rotulos = por_lote.index.str.replace("lote_", "", regex=False)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(rotulos, por_lote["reservas"], color="#cfd6e3")
ax.set_ylabel("reservas no lote")
ax.tick_params(axis="x", rotation=70)
ax2 = ax.twinx()
ax2.plot(rotulos, por_lote["taxa"], color="#7a1f3d", marker="o")
ax2.set_ylabel("taxa de cancelamento")
ax2.set_ylim(0, 0.6)
ax.set_title("Volume e taxa de cancelamento por lote (após o ETL)")
fig.tight_layout()
fig.savefig(evid_dir / f"cancelamento_por_lote_{cycle_id}.png", dpi=110)
plt.show()

## 4. Partição cronológica
Teste = últimos `teste_fracao` dos dados disponíveis **por data de chegada**, cortando em fronteira de dia. Ele fica reservado à avaliação final; seleção de features, competição e promoção usam validação cruzada cronológica dentro do dev.

In [ ]:
dev, teste = dividir_temporal(df, cfg["avaliacao"]["teste_fracao"], cfg["avaliacao"]["coluna_tempo"])

assert dev["arrival_date"].max() < teste["arrival_date"].min(), "dev e teste se sobrepõem no tempo"
assert not set(zip(dev["lote_id"], dev["linha_origem"])) & set(zip(teste["lote_id"], teste["linha_origem"])), "registro nos dois conjuntos"
minimo = cfg["avaliacao"]["min_linhas_dev"]
assert len(dev) >= minimo, f"dev com {len(dev)} linhas; mínimo exigido {minimo}"

pd.DataFrame({
    "linhas": [len(dev), len(teste)],
    "início": [dev["arrival_date"].min().date(), teste["arrival_date"].min().date()],
    "fim": [dev["arrival_date"].max().date(), teste["arrival_date"].max().date()],
    "taxa de cancelamento": [dev[alvo].mean().round(3), teste[alvo].mean().round(3)],
}, index=["dev", "teste"])

## 5. Pré-processador sem vazamento
`construir_preprocessador` devolve um `ColumnTransformer` **não ajustado**. Aqui ele é ajustado só no dev e apenas aplicado ao teste. Nos notebooks seguintes ele entra num `Pipeline`, para a validação cruzada reajustá-lo a cada fold.

In [ ]:
feats = lista_features(cfg)
pre = construir_preprocessador(cfg)
X_dev = pre.fit_transform(dev[feats])      # ajuste somente no dev
X_teste = pre.transform(teste[feats])      # teste só é transformado

assert list(X_dev.columns) == list(X_teste.columns)
print(f"X_dev {X_dev.shape} | X_teste {X_teste.shape} | ausentes: {int(X_dev.isna().sum().sum())} / {int(X_teste.isna().sum().sum())}")
print({grupo: len(cols) for grupo, cols in colunas_modelagem(cfg).items()})

w = pre.named_transformers_["num_cauda"].named_steps["winsor"]
nomes = list(w.get_feature_names_out())
limites = pd.DataFrame({
    "limite_sup (aprendido no dev)": w.limite_sup_,
    "máx no dev": dev[nomes].max().to_numpy(),
    "máx no teste": teste[nomes].max().to_numpy(),
}, index=nomes).loc[["lead_time", "adr", "adults", "days_in_waiting_list"]]
limites

## 6. Gravação dos artefatos e do contexto do ciclo

In [ ]:
caminhos = salvar_particoes(dev, teste, cfg, RAIZ)
rel_path = lambda p: str(Path(p).relative_to(RAIZ))
agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

caminho_diag = evid_dir / f"diagnostico_{cycle_id}.json"
caminho_diag.write_text(json.dumps({
    "cycle_id": cycle_id, "data_version": data_version, "gerado_em": agora,
    "lotes": lotes_atuais, "lotes_novos_vs_ciclo_anterior": lotes_novos,
    "antes": diag_antes, "depois": diag_depois, "relatorio_etl": rel,
}, indent=2, ensure_ascii=False), encoding="utf-8")

ctx = {
    "cycle_id": cycle_id,
    "data_version": data_version,
    "criado_em": agora,
    "n_lotes": len(lotes_atuais),
    "lotes_novos_vs_ciclo_anterior": lotes_novos,
    "periodo": {"chegada_min": str(df["arrival_date"].min().date()), "chegada_max": str(df["arrival_date"].max().date())},
    "particao": {
        "teste_fracao_pedida": cfg["avaliacao"]["teste_fracao"],
        "dev_n": len(dev), "teste_n": len(teste),
        "dev_fim": str(dev["arrival_date"].max().date()), "teste_inicio": str(teste["arrival_date"].min().date()),
    },
    "artefatos": {"dev": rel_path(caminhos["dev"]), "teste": rel_path(caminhos["teste"]), "diagnostico": rel_path(caminho_diag)},
    "colunas": {"alvo": alvo, "tempo": cfg["avaliacao"]["coluna_tempo"], "features": feats, **colunas_modelagem(cfg)},
    "etl": rel,
}
salvar_contexto(ctx, cfg, RAIZ)
print("contexto :", rel_path(RAIZ / cfg["dados"]["dir_processed"] / "ciclo_atual.json"))
print("dev/teste:", rel_path(caminhos["dev"]), "|", rel_path(caminhos["teste"]))
print("evidência:", rel_path(caminho_diag))

## Decisões e limitações
- **Momento da previsão:** no ato da reserva. Por isso saem `assigned_room_type` e `booking_changes`, que só existem depois.
- **`total_of_special_requests` e `required_car_parking_spaces`** ficam: o dataset não informa quando foram registrados, então assumimos que fazem parte da reserva. É uma suposição e está documentada.
- **Rótulo:** assumimos que o desfecho (cancelou ou não) é conhecido até a data de chegada, e usamos essa data como eixo do tempo.
- **Escopo:** dois hotéis de Portugal (2015-2017). Não generaliza para outros mercados nem para o período pós-pandemia.
- **Próximo passo:** o notebook 02 lê `data/processed/ciclo_atual.json`.